# SHAP: Explaining an AI-Set Insurance Price

## The situation

You are the **Pricing & Underwriting Lead** at an auto insurance company. Your team's
machine-learning model looks at each policyholder's driving profile and predicts their
**expected annual claim cost** in dollars — the number the company uses to set that
person's renewal premium.

The model is accurate. That was never the hard part. The hard part is this: auto
insurance pricing is one of the most heavily regulated uses of machine learning there
is. Regulators can (and do) ask insurers to justify *why* a specific customer was priced
the way they were — and in many places, pricing on things like gender, nationality, or
ethnicity is restricted or banned outright, directly or through a proxy for them.

Before this model goes anywhere near a regulatory review, you need to be able to answer,
for any one customer: **why did the model say that?** And zooming out: **is there a
pattern across the whole portfolio that shouldn't be there?**

That is exactly what **SHAP** (SHapley Additive exPlanations) is for. In this notebook
you will:

1. Build the core intuition behind SHAP with no machine learning at all — just a team
   fairly splitting credit for a shared win.
2. Meet the pricing model and one specific policyholder, "Jordan."
3. Use SHAP to answer, in plain numbers, *why* Jordan was priced the way he was.
4. Zoom out from Jordan to the whole portfolio and check whether what you found is a
   one-off or a systemic problem.
5. Apply the same skill to a very different, equally regulated use of AI: screening
   resumes for a job.

No formulas required — just plots, and how to read them.

In [ ]:
#@title 🗺️ Roadmap: from a fair bonus split to auditing an AI price { display-mode: "form" }
from IPython.display import HTML, display

_STEPS = [("🤝", "Fair Credit", "splitting a bonus by contribution"),
          ("🚗", "Meet the Model", "pricing Jordan's policy"),
          ("🧮", "SHAP Values", "the fair split, applied to a prediction"),
          ("💧", "Waterfall", "why isn't Jordan's record paying off?"),
          ("🐝", "Beeswarm", "is it just Jordan, or everyone?"),
          ("🧭", "Apply It", "hiring & compliance")]
_GRAD = ["#667eea", "#7b71ee", "#9a6fe2", "#b06fd0", "#c56fbe", "#db6fa9"]

_blocks = ""
for (ic, t, d), g in zip(_STEPS, _GRAD):
    _blocks += (
        '<div class="rm-step"><div class="rm-ic" style="background:linear-gradient(135deg,'
        + g + "," + g + 'cc)">' + ic + '</div>'
        '<div class="rm-t">' + t + '</div><div class="rm-q">' + d + '</div></div>'
    )

_HTML = r"""
<style>
.rm{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border-radius:18px;padding:20px 16px;margin:8px 0;border:1px solid #ecebff}
.rm-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 4px}
.rm-s{font-size:12px;color:#6b6685;margin:0 0 16px}
.rm-row{display:flex;align-items:stretch;flex-wrap:wrap;gap:0}
.rm-step{flex:1 1 150px;min-width:150px;text-align:center;padding:0 6px}
.rm-ic{width:50px;height:50px;border-radius:50%;margin:0 auto 8px;display:flex;align-items:center;
       justify-content:center;font-size:22px;color:#fff;box-shadow:0 6px 14px rgba(102,126,234,.35)}
.rm-t{font-weight:800;font-size:13px;color:#2c2350}
.rm-q{font-size:11px;color:#6b6685;margin-top:3px;line-height:1.35}
</style>
<div class="rm">
  <div class="rm-h">Roadmap</div>
  <div class="rm-s">From a fair bonus split to auditing an AI-driven insurance price.</div>
  <div class="rm-row">__BLOCKS__</div>
</div>
"""

display(HTML(_HTML.replace("__BLOCKS__", _blocks)))


In [ ]:
#@title Setup { display-mode: "form" }
!pip install -q shap scikit-learn

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import permutations
from IPython.display import HTML, display

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

PURPLE, PURPLE2 = '#667eea', '#764ba2'
GREEN, RED, AMBER = '#39b36a', '#e0796d', '#e0a23c'
RANDOM_STATE = 0

import shap
print('ready')


## 1. Fair credit for a team win

Before touching any model, here is the idea SHAP is built on — and it has nothing to do
with machine learning.

**The situation.** Three colleagues, Ana, Ben, and Cleo, jointly land a **\$1,000,000**
deal. Alone, each of them could have closed a *smaller* version of it — and in different
pairs, they close different amounts too. Here is what the deal would have been worth for
every possible combination of the three of them:

| Team | Deal value |
|---|---|
| *(no one)* | \$0 |
| Ana alone | \$150,000 |
| Ben alone | \$100,000 |
| Cleo alone | \$80,000 |
| Ana + Ben | \$500,000 |
| Ana + Cleo | \$400,000 |
| Ben + Cleo | \$300,000 |
| Ana + Ben + Cleo | \$1,000,000 |

**The question.** How should the \$1,000,000 bonus be split so that each person gets
credit for what they actually contributed?

Splitting it equally (\$333,333 each) ignores this table entirely — it would pay Cleo the
same as Ana, even though Ana is worth almost twice as much solo and lifts every pairing
she's in by more. We need a split that actually looks at *marginal contribution*: how
much value a person adds to whatever group they join.

### 🎯 Task — compute each person's marginal contribution

Here's the trick: a person's contribution depends on who was *already in the room* when
they joined. If Ana joins first, she gets credit for going from \$0 to \$150,000. If she
joins *last*, after Ben and Cleo already closed \$300,000 together, her marginal
contribution is \$1,000,000 − \$300,000 = \$700,000 — much bigger, because she's completing
a deal that was already large.

Neither number is "the" answer. The fair thing to do is: look at **every possible
joining order**, work out each person's marginal contribution in that order, and then
**average** each person's contribution across all the orders. That average is the fair
split — and it happens to be exactly what a Shapley value is, no formula needed.

In [ ]:
# 🎯 YOUR TURN — Exercise 1: turn marginal contributions into a fair split.
#
# 💭 Think first: if Ana joins first, her contribution is $150,000 (0 -> 150k). If she
#    joins last, after Ben and Cleo already closed $300,000 together, her contribution
#    is $700,000 (300k -> 1M). Both are "correct" for that one order -- so what single
#    number would you report as *the* fair value of what Ana brings to the deal?
deal_value = {
    frozenset(): 0,
    frozenset({'Ana'}): 150_000,
    frozenset({'Ben'}): 100_000,
    frozenset({'Cleo'}): 80_000,
    frozenset({'Ana', 'Ben'}): 500_000,
    frozenset({'Ana', 'Cleo'}): 400_000,
    frozenset({'Ben', 'Cleo'}): 300_000,
    frozenset({'Ana', 'Ben', 'Cleo'}): 1_000_000,
}

people = ['Ana', 'Ben', 'Cleo']
contributions = {p: [] for p in people}

print(f"{'order':<18}{'Ana':>10}{'Ben':>10}{'Cleo':>10}")
for order in permutations(people):
    joined = frozenset()
    running_value = deal_value[joined]
    row = {}
    for person in order:
        joined = joined | {person}
        new_value = deal_value[joined]
        # 🎯 Implement: subtract `running_value` from `new_value` -- that difference is
        # this person's marginal contribution in THIS order. Store it in `marginal`.
        marginal = ...
        contributions[person].append(marginal)
        row[person] = marginal
        running_value = new_value
    print(f"{' -> '.join(order):<18}{row.get('Ana', ''):>10,}{row.get('Ben', ''):>10,}{row.get('Cleo', ''):>10,}")

# 🎯 Implement: build a dict comprehension over `contributions.items()` that maps each
# person `p` to `np.mean(v)` of their list of marginal contributions `v`. Store it in
# `fair_split` (same shape as `equal_split` below: {person: dollar amount}).
fair_split = ...
equal_split = {p: 1_000_000 / 3 for p in people}

print('\nFair (Shapley) split, averaged across all 6 orders:')
for p in people:
    print(f'  {p:<6} ${fair_split[p]:,.0f}')
print(f"\nCheck -- do the fair shares add back up to the full deal? "
      f"${sum(fair_split.values()):,.0f}")


In [ ]:
#@title 📊 Visualization: equal split vs. the fair (Shapley) split (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

def _split_bars(fair_split, equal_split):
    people = ['Ana', 'Ben', 'Cleo']
    solo = {'Ana': 150_000, 'Ben': 100_000, 'Cleo': 80_000}
    mx = max(max(fair_split.values()), max(equal_split.values())) * 1.15
    rows = ""
    for p in people:
        f_pct = fair_split[p] / mx * 100
        e_pct = equal_split[p] / mx * 100
        rows += (
            '<div class="sb-row"><div class="sb-name">' + p
            + '<div class="sb-solo">solo value: $' + format(solo[p], ",") + '</div></div>'
            '<div class="sb-bars">'
            '<div class="sb-track"><div class="sb-fill sb-equal" style="width:' + f"{e_pct:.0f}" + '%"></div>'
            '<div class="sb-lab">equal: $' + format(round(equal_split[p]), ",") + '</div></div>'
            '<div class="sb-track"><div class="sb-fill sb-fair" style="width:' + f"{f_pct:.0f}" + '%"></div>'
            '<div class="sb-lab">fair: $' + format(round(fair_split[p]), ",") + '</div></div>'
            '</div></div>'
        )
    return rows

_HTML = r"""
<style>
.sb{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:720px;color:#2c2350}
.sb-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.sb-s{font-size:12px;color:#6b6685;margin:0 0 16px;line-height:1.5}
.sb-row{display:flex;gap:14px;align-items:center;padding:9px 0;border-top:1px solid #eceaf6}
.sb-row:first-of-type{border-top:none}
.sb-name{flex:0 0 90px;font-weight:800;font-size:13.5px;color:#2c2350}
.sb-solo{font-weight:400;font-size:10.5px;color:#8b86a3;margin-top:2px}
.sb-bars{flex:1 1 auto;display:flex;flex-direction:column;gap:5px}
.sb-track{position:relative;background:#fff;border:1px solid #e6e8ee;border-radius:8px;height:22px}
.sb-fill{height:100%;border-radius:7px}
.sb-equal{background:#b8b4c8}
.sb-fair{background:linear-gradient(90deg,#667eea,#764ba2)}
.sb-lab{position:absolute;left:8px;top:2px;font-size:10.5px;font-weight:700;color:#2c2350}
</style>
<div class="sb">
  <div class="sb-h">Equal split vs. fair split</div>
  <div class="sb-s">Equal split ignores how much each person actually moved the deal. The fair split
  pays Ana more because she is the strongest solo closer and lifts every pairing she joins &mdash;
  and pays Cleo less, not because she did nothing, but because her marginal impact is
  consistently smaller.</div>
  __ROWS__
</div>
"""
display(HTML(_HTML.replace("__ROWS__", _split_bars(fair_split, equal_split))))


In [ ]:
#@title 🎯 Quick check: why does Ana get the biggest share? (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
.q1{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:700px;color:#2c2350}
.q1-h{font-size:15.5px;font-weight:800;color:#3b2d6b;margin:0 0 8px}
.q1-q{font-size:13px;color:#4a4360;line-height:1.45;margin-bottom:12px}
.q1-opt{background:#fff;border:1.5px solid #e6e8ee;border-radius:12px;padding:10px 13px;
        margin-bottom:8px;font-size:12.5px;color:#3a3450;line-height:1.4;cursor:pointer;transition:border-color .15s}
.q1-opt:hover{border-color:#c9b8ec}
.q1-correct{border-color:#39b36a;background:#eef9f1}
.q1-wrong{border-color:#e0796d;background:#fdf1ef}
.q1-fb{margin-top:10px;padding:10px 13px;border-radius:12px;font-size:12.5px;line-height:1.45}
.q1-fb-good{background:#eef9f1;color:#1f6b3e;border:1px solid #b9e6c7}
.q1-fb-bad{background:#fdf1ef;color:#8a3d34;border:1px solid #f0c4bd}
</style>
<div class="q1">
  <div class="q1-h">🎯 Quick check</div>
  <div class="q1-q">Why does Ana end up with the largest fair (Shapley) share &mdash; larger than
  you'd expect from her solo deal value alone?</div>
  <div class="q1-opt" id="q1-opt-0" onclick="q1pick(0)">Her fair share is her solo value plus a fixed bonus for being the one who "started" the deal.</div>
  <div class="q1-opt" id="q1-opt-1" onclick="q1pick(1)">Averaged across every joining order, Ana's presence tends to add more value to whatever group she joins than Ben's or Cleo's does &mdash; not just in the one order where she happens to close it alone.</div>
  <div class="q1-opt" id="q1-opt-2" onclick="q1pick(2)">In the order where she joins last, her marginal contribution ($700,000) is the single biggest number in the whole table &mdash; and that's the number that decided her share.</div>
  <div class="q1-opt" id="q1-opt-3" onclick="q1pick(3)">The split is proportional to each person's solo deal value, scaled up so the three shares add to $1,000,000.</div>
  <div class="q1-fb" id="q1-fb" style="display:none"></div>
</div>
<script>
(function(){
  var correct = 1;
  var feedback = [
    "Not quite — nothing in the calculation gives special credit for “starting” the deal; every order is weighted equally.",
    "Right — Shapley averages marginal contributions across every order. Ana's average is bigger because she tends to add more value in combination with others, not just alone.",
    "Close, but not the mechanism — $700,000 is real, but it's just one of six orders. Shapley averages all of them, it doesn't pick the best case for anyone.",
    "That's a different rule (splitting proportional to solo value) — it would actually give Ana $454,545, not her real share of $403,333. Solo value alone isn't what Shapley measures."
  ];
  window.q1pick = function(i){
    for(var j=0;j<4;j++){
      var el = document.getElementById('q1-opt-'+j);
      el.classList.remove('q1-correct','q1-wrong');
      if(j===correct) el.classList.add('q1-correct');
    }
    if(i!==correct) document.getElementById('q1-opt-'+i).classList.add('q1-wrong');
    var fb = document.getElementById('q1-fb');
    fb.style.display = 'block';
    fb.textContent = feedback[i];
    fb.className = 'q1-fb ' + (i===correct ? 'q1-fb-good' : 'q1-fb-bad');
  };
})();
</script>
"""))


**The bridge to SHAP.** Swap the metaphor:

- teammates &rarr; **features** of a policyholder (age, mileage, prior claims, ...)
- the deal value &rarr; the **model's prediction**
- the order people join &rarr; the order you **reveal features** to the model, one at a
  time, starting from "knowing nothing"

SHAP computes, for one prediction, exactly what you just computed by hand: each
feature's *average marginal contribution*, across every possible order you could have
revealed the features in. That number is the feature's **SHAP value** — and just like
the bonus split, all the SHAP values for one prediction always add up *exactly* to the
gap between "knowing nothing" and the model's actual prediction for that person. Nothing
is left over, nothing is double-counted.

## 2. Meet the model and Jordan

Your team's model predicts `expected_annual_claim_cost` — in dollars — for each
policyholder, from their driving profile:

- `age`, `years_licensed` — younger and less experienced drivers cost more, on average.
- `annual_mileage` — more driving, more exposure to risk.
- `vehicle_value` — pricier vehicles cost more to repair or replace.
- `prior_at_fault_claims_5y` — a direct, strong signal.
- `urban_vs_rural` — more traffic, more accidents.
- `background_group` — an anonymized demographic category the company has historically
  recorded. There is no actuarial reason a person's background should affect how much
  they cost to insure — but it is in the historical data, and any model trained on that
  data is free to pick it up if it happens to be predictive. This is precisely what your
  compliance review needs to catch, if it's there.

In [ ]:
#@title Load the insurance dataset (offline builder) { display-mode: "form" }
# Reproduces generate_insurance_claims_dataset.py exactly, so this notebook has no
# dependency on the repo being cloned -- see that script for the full design rationale.
BACKGROUND_GROUPS = ["Group A", "Group B", "Group C"]
BIASED_GROUP = "Group C"
BIAS_AMOUNT = 300.0
BASE_COST = 500.0
MIN_COST = 150.0

def make_insurance_dataset(n_rows=3500, seed=42):
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 80, n_rows).astype(float)
    years_licensed = np.clip(age - 18 - rng.integers(0, 6, n_rows), 0, None).astype(float)
    annual_mileage = np.clip(rng.normal(12000, 5000, n_rows), 1000, None)
    vehicle_value = np.clip(rng.normal(28000, 11000, n_rows), 4000, None)
    prior_at_fault_claims_5y = rng.choice([0, 1, 2, 3, 4], size=n_rows, p=[0.55, 0.25, 0.12, 0.06, 0.02]).astype(float)
    urban_vs_rural = rng.choice(['urban', 'rural'], size=n_rows, p=[0.6, 0.4])
    background_group = rng.choice(BACKGROUND_GROUPS, size=n_rows, p=[0.4, 0.35, 0.25])

    effect_age = np.clip(50.0 - age, 0, None) * 7.0
    effect_experience = np.clip(10.0 - years_licensed, 0, None) * 15.0
    effect_mileage = (annual_mileage - 12000.0) / 1000.0 * 10.0
    effect_vehicle = vehicle_value * 0.01
    effect_claims = prior_at_fault_claims_5y * 220.0
    effect_urban = np.where(urban_vs_rural == 'urban', 150.0, 0.0)
    effect_background = np.where(background_group == BIASED_GROUP, BIAS_AMOUNT, 0.0)
    noise = rng.normal(0.0, 80.0, n_rows)

    cost = np.clip(BASE_COST + effect_age + effect_experience + effect_mileage + effect_vehicle
                   + effect_claims + effect_urban + effect_background + noise, MIN_COST, None)

    return pd.DataFrame({
        'age': age.astype(int), 'years_licensed': years_licensed.astype(int),
        'annual_mileage': annual_mileage.round(0).astype(int), 'vehicle_value': vehicle_value.round(0).astype(int),
        'prior_at_fault_claims_5y': prior_at_fault_claims_5y.astype(int), 'urban_vs_rural': urban_vs_rural,
        'background_group': background_group, 'expected_annual_claim_cost': cost.round(2),
    })

df = make_insurance_dataset(n_rows=3500, seed=RANDOM_STATE)
print(f'{len(df)} policyholders')
df.head()


**The model itself.** We'll train a **random forest**: instead of one decision tree,
it grows a few hundred slightly different trees, each one making its own guess at a
policyholder's cost from a randomly varied subset of the data, and then averages all of
their guesses into one final price. This is exactly why the model can be so hard to
"read" directly — the actual prediction is a blend of hundreds of trees, not one clean
rule you could just look at — and exactly why it's a *tree-based* model, which matters
in a moment: it's what lets `shap` compute exact SHAP values quickly (TreeSHAP), instead
of the slow, brute-force approach the fair-split exercise used.

In [ ]:
#@title Prepare features and train the pricing model { display-mode: "form" }
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X = df.copy()
X['is_urban'] = (X['urban_vs_rural'] == 'urban').astype(int)
X['background_group_code'] = X['background_group'].map({'Group A': 0, 'Group B': 1, 'Group C': 2})

FEATURES = ['age', 'years_licensed', 'annual_mileage', 'vehicle_value',
            'prior_at_fault_claims_5y', 'is_urban', 'background_group_code']
FEATURE_LABELS = {'is_urban': 'urban_vs_rural', 'background_group_code': 'background_group'}

y = X['expected_annual_claim_cost']
X_train, X_test, y_train, y_test = train_test_split(
    X[FEATURES], y, test_size=0.25, random_state=RANDOM_STATE)
X_train = X_train.rename(columns=FEATURE_LABELS)
X_test = X_test.rename(columns=FEATURE_LABELS)

model = RandomForestRegressor(n_estimators=300, max_depth=8, n_jobs=-1, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

mae = mean_absolute_error(y_test, model.predict(X_test))
print(f'On policyholders the model has never seen, its predictions are off by ${mae:,.0f} on average '
      f'(portfolio mean cost: ${y.mean():,.0f}).')


In [ ]:
#@title 🧑 Meet Jordan { display-mode: "form" }
test_df = df.loc[X_test.index].copy()
test_df['predicted_cost'] = model.predict(X_test)
portfolio_avg = test_df.predicted_cost.mean()

# an otherwise unremarkable profile: no prior claims, decades of experience (age >= 50
# also guarantees years_licensed is well past the experience discount cutoff), and
# mileage/vehicle value both near the middle of the portfolio.
candidates = test_df[
    (test_df.background_group == BIASED_GROUP)
    & (test_df.prior_at_fault_claims_5y == 0)
    & (test_df.age >= 50)
    & test_df.annual_mileage.between(*df.annual_mileage.quantile([0.35, 0.65]))
    & test_df.vehicle_value.between(*df.vehicle_value.quantile([0.35, 0.65]))
]
jordan_row = candidates.sort_values('predicted_cost', ascending=False).iloc[0]
jordan_idx = X_test.index.get_loc(jordan_row.name)
gap = jordan_row.predicted_cost - portfolio_avg

from IPython.display import HTML, display
_HTML = r"""
<style>
.jc{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:640px;color:#2c2350}
.jc-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 10px}
.jc-grid{display:flex;flex-wrap:wrap;gap:9px;margin-bottom:14px}
.jc-cell{flex:1 1 150px;background:#fff;border:1px solid #e6e8ee;border-radius:12px;padding:9px 12px}
.jc-lab{font-size:10.5px;color:#8b86a3;text-transform:uppercase;letter-spacing:.03em}
.jc-val{font-size:14.5px;font-weight:800;color:#2c2350;margin-top:2px}
.jc-pred{display:flex;gap:14px;align-items:center;background:#fff;border:1.5px solid #c9b8ec;
         border-radius:12px;padding:12px 16px}
.jc-pv{font-size:22px;font-weight:800;color:#764ba2}
.jc-pl{font-size:11.5px;color:#6b6685;line-height:1.4}
</style>
<div class="jc">
  <div class="jc-h">Jordan &mdash; policyholder up for renewal</div>
  <div class="jc-grid">
    <div class="jc-cell"><div class="jc-lab">Age</div><div class="jc-val">__AGE__</div></div>
    <div class="jc-cell"><div class="jc-lab">Years licensed</div><div class="jc-val">__YL__</div></div>
    <div class="jc-cell"><div class="jc-lab">Annual mileage</div><div class="jc-val">__MI__ mi</div></div>
    <div class="jc-cell"><div class="jc-lab">Vehicle value</div><div class="jc-val">$__VV__</div></div>
    <div class="jc-cell"><div class="jc-lab">Prior at-fault claims (5y)</div><div class="jc-val">__PC__</div></div>
    <div class="jc-cell"><div class="jc-lab">Area</div><div class="jc-val">__UR__</div></div>
  </div>
  <div class="jc-pred">
    <div class="jc-pv">$__PRED__</div>
    <div class="jc-pl">the model's predicted expected annual claim cost for Jordan &mdash;
    about <b>$__GAP__ above</b> the portfolio average of $__AVG__. His record above looks
    fine, arguably better than average: decades of experience, zero at-fault claims.
    So why the gap?</div>
  </div>
</div>
"""
display(HTML(_HTML
    .replace("__AGE__", str(int(jordan_row.age)))
    .replace("__YL__", str(int(jordan_row.years_licensed)))
    .replace("__MI__", f"{int(jordan_row.annual_mileage):,}")
    .replace("__VV__", f"{int(jordan_row.vehicle_value):,}")
    .replace("__PC__", str(int(jordan_row.prior_at_fault_claims_5y)))
    .replace("__UR__", jordan_row.urban_vs_rural)
    .replace("__PRED__", f"{jordan_row.predicted_cost:,.0f}")
    .replace("__GAP__", f"{gap:,.0f}")
    .replace("__AVG__", f"{portfolio_avg:,.0f}")
))


## 3. Computing SHAP values

Time to do for Jordan's prediction exactly what you did for the \$1,000,000 deal: work
out each feature's average marginal contribution to his predicted cost, across every
order the model could "learn about" his features.

For a tree-based model like our `RandomForestRegressor`, the `shap` library offers
**TreeSHAP**: an exact, fast method that doesn't actually need to try every ordering by
brute force (that would be too slow), but computes precisely the same
average-marginal-contribution numbers you computed by hand above.

### 🎯 Task — compute Jordan's SHAP values

`shap.TreeExplainer` can get the baseline two different ways: `feature_perturbation=
'interventional'` (the default) estimates it from a random background sample, which adds
sampling noise every time you re-run it. `feature_perturbation='tree_path_dependent'`
instead reads it straight off the trees' own training-sample statistics — exact and
deterministic, no background sample needed. Use the second one.

In [ ]:
# 🎯 YOUR TURN — Exercise 2: compute Jordan's SHAP values.
#
# 💭 Think first: our model is a RandomForestRegressor -- a tree ensemble. Which SHAP
#    algorithm mentioned in Part 3's intro is built specifically for tree-based models,
#    letting it skip trying every ordering by brute force?
#
# 🎯 Implement: call `shap.TreeExplainer(model, feature_perturbation='tree_path_dependent')`
#    (see the task above) and store the result in `explainer`. Then call `explainer(X_test)`
#    -- explainers are callable -- and store the returned Explanation object in `sv`.
explainer = ...
sv = ...

# show the readable category labels (e.g. "Group C", "urban") on the plots below, instead
# of the numeric codes the model actually trains on -- doesn't change any SHAP value.
display_vals = X_test.copy().astype(object)
display_vals['urban_vs_rural'] = df.loc[X_test.index, 'urban_vs_rural'].values
display_vals['background_group'] = df.loc[X_test.index, 'background_group'].values
sv.display_data = display_vals.values

baseline = sv.base_values[0]
print(f'Baseline (predicted cost knowing nothing about the policyholder): ${baseline:,.0f}')
print(f"Jordan's SHAP values add up, exactly, to his prediction:")
print(f'  baseline (${baseline:,.0f}) + sum of SHAP values (${sv.values[jordan_idx].sum():,.0f})'
      f' = ${baseline + sv.values[jordan_idx].sum():,.0f}')
print(f"  model's actual prediction for Jordan: ${jordan_row.predicted_cost:,.0f}")


### ⭐ Bonus &mdash; the formula behind all of this

Skippable — everything after this point stands on its own without it. But if you want to
see the rule you already ran as code in Part 1, written out symbolically, here it is.

For a feature (or a person, in the bonus-split example) $i$, out of the full set of
features $F$:

$$\phi_i = \frac{1}{|F|!} \sum_{\pi} \Big[ v\big(S_i^\pi \cup \{i\}\big) - v\big(S_i^\pi\big) \Big]$$

Reading it left to right against the code you already wrote for `deal_value` and
`contributions`:

- $\pi$ is one ordering — one row of the `for order in permutations(people)` loop.
- $S_i^\pi$ is "who's already in the room" when $i$ joins in that ordering — the
  running `joined` set, *before* adding $i$.
- $v(S)$ is the value of a coalition $S$ — `deal_value[S]` in the toy example, or the
  model's expected prediction knowing only the features in $S$, here.
- the term in brackets is one marginal contribution — one `marginal` in the code.
- summing over every ordering $\pi$ and dividing by $|F|!$ (the number of orderings) is
  exactly the `np.mean(v)` that produced `fair_split`.

$\phi_i$ — the average marginal contribution — *is* the Shapley value, and applied to a
model instead of a deal, it's the SHAP value plotted in the waterfall below. TreeSHAP
doesn't actually loop over `|F|!` orderings the way our toy example's code does (for
Jordan's 7 features that would be $7! = 5{,}040$ orderings, and real models have far more
features than that) — it uses the tree-weighted shortcut from a moment ago to land on the
exact same numbers in a fraction of the time.

## 4. Waterfall plot &mdash; why isn't Jordan's good record paying off?

A **waterfall plot** explains *one* prediction. It reads top to bottom:

- it starts at the **baseline** — the average price the model learned to assign across
  the policyholders it was trained on, before it knows anything about *this* particular
  person (a little different from, but close to, the test-set portfolio average you saw
  on Jordan's card above — different group of policyholders, same idea);
- each **bar** is one feature, sized by its SHAP value: a **red** bar pushes the
  prediction *up*, a **blue** bar pushes it *down*;
- the bars stack in order of size, and the bottom of the chart lands **exactly** on the
  model's actual prediction for that person — the same "no leftovers" guarantee from the
  bonus split.

In [ ]:
#@title 📊 How to read a waterfall plot (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
.wl{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border:1px solid #ecebff;border-radius:18px;padding:16px 18px;max-width:700px;color:#2c2350}
.wl-h{font-size:15.5px;font-weight:800;color:#3b2d6b;margin:0 0 10px}
.wl-row{display:flex;gap:11px;align-items:center;padding:5px 0}
.wl-sw{flex:0 0 34px;height:16px;border-radius:5px}
.wl-tx{font-size:12px;color:#4a4360;line-height:1.4}
.wl-tx b{color:#2c2350}
</style>
<div class="wl">
  <div class="wl-h">Reading a SHAP waterfall plot</div>
  <div class="wl-row"><div class="wl-sw" style="background:#9aa0b5"></div>
    <div class="wl-tx"><b>Top: E[f(x)]</b> &mdash; the baseline. The model's average prediction, before it knows anything about this specific person.</div></div>
  <div class="wl-row"><div class="wl-sw" style="background:#e0796d"></div>
    <div class="wl-tx"><b>Red bars</b> &mdash; this feature's value pushed the prediction <b>up</b>.</div></div>
  <div class="wl-row"><div class="wl-sw" style="background:#4d8fd6"></div>
    <div class="wl-tx"><b>Blue bars</b> &mdash; this feature's value pushed the prediction <b>down</b>.</div></div>
  <div class="wl-row"><div class="wl-sw" style="background:#2c2350"></div>
    <div class="wl-tx"><b>Bottom: f(x)</b> &mdash; the model's final prediction for this person. Baseline + every bar, exactly.</div></div>
</div>
"""))


In [ ]:
#@title Jordan's waterfall plot { display-mode: "form" }
shap.plots.waterfall(sv[jordan_idx])


In [ ]:
#@title 🎯 Quick check: which 3 bars matter most? (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
.q2{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:700px;color:#2c2350}
.q2-h{font-size:15.5px;font-weight:800;color:#3b2d6b;margin:0 0 8px}
.q2-q{font-size:13px;color:#4a4360;line-height:1.45;margin-bottom:12px}
.q2-opt{display:flex;align-items:center;gap:9px;background:#fff;border:1.5px solid #e6e8ee;
        border-radius:12px;padding:9px 13px;margin-bottom:7px;font-size:12.5px;color:#3a3450;cursor:pointer}
.q2-opt input{width:15px;height:15px;flex:0 0 auto}
.q2-good{border-color:#39b36a;background:#eef9f1}
.q2-bad{border-color:#e0796d;background:#fdf1ef}
.q2-missed{border-color:#e0a23c;border-style:dashed;background:#fdf7ec}
.q2-btn{background:#764ba2;color:#fff;border:none;border-radius:10px;padding:8px 16px;
        font-size:12.5px;font-weight:700;cursor:pointer;margin-top:4px}
.q2-btn:hover{background:#663d94}
.q2-fb{margin-top:10px;padding:10px 13px;border-radius:12px;font-size:12.5px;line-height:1.45;
       background:#f3f0fb;color:#3b2d6b;border:1px solid #dcd3f0}
</style>
<div class="q2">
  <div class="q2-h">🎯 Quick check &mdash; select exactly 3</div>
  <div class="q2-q">Jordan's waterfall breaks his price into 7 feature bars. Select the <b>3 bars</b>
  &mdash; pushing in either direction &mdash; doing the most work in setting his final price.</div>
  <label class="q2-opt"><input type="checkbox" id="q2-cb-age"> Age</label>
  <label class="q2-opt"><input type="checkbox" id="q2-cb-yl"> Years licensed</label>
  <label class="q2-opt"><input type="checkbox" id="q2-cb-mi"> Annual mileage</label>
  <label class="q2-opt"><input type="checkbox" id="q2-cb-vv"> Vehicle value</label>
  <label class="q2-opt"><input type="checkbox" id="q2-cb-pc"> Prior at-fault claims (5y)</label>
  <label class="q2-opt"><input type="checkbox" id="q2-cb-ur"> Urban vs. rural</label>
  <label class="q2-opt"><input type="checkbox" id="q2-cb-bg"> Background group</label>
  <div><button class="q2-btn" onclick="q2check()">Check my answer</button></div>
  <div class="q2-fb" id="q2-fb" style="display:none"></div>
</div>
<script>
(function(){
  var correctKeys = ['bg','pc','ur'];
  var allKeys = ['age','yl','mi','vv','pc','ur','bg'];
  window.q2check = function(){
    allKeys.forEach(function(k){
      var cb = document.getElementById('q2-cb-'+k);
      var label = cb.closest('label');
      label.classList.remove('q2-good','q2-bad','q2-missed');
      var checked = cb.checked;
      var isCorrect = correctKeys.indexOf(k) !== -1;
      if(checked && isCorrect) label.classList.add('q2-good');
      else if(checked && !isCorrect) label.classList.add('q2-bad');
      else if(!checked && isCorrect) label.classList.add('q2-missed');
    });
    var fb = document.getElementById('q2-fb');
    fb.style.display = 'block';
    fb.innerHTML = "<b>Background group</b> (+$235), <b>Prior at-fault claims (5y)</b> (−$163), and <b>Urban vs. rural</b> (+$57) are the three biggest movers on Jordan's row — every other feature stays under $30 in either direction.";
  };
})();
</script>
"""))


In [ ]:
#@title What if we take background_group out of the picture? { display-mode: "form" }
jordan_vals = pd.Series(sv.values[jordan_idx], index=X_test.columns)
without_bg = sv.base_values[jordan_idx] + jordan_vals.drop('background_group').sum()

print(f"Jordan's actual predicted price:            ${jordan_row.predicted_cost:,.0f}")
print(f"background_group's own contribution:         {jordan_vals['background_group']:+,.0f}")
print(f"predicted price WITHOUT background_group:    ${without_bg:,.0f}")
print(f"portfolio average predicted price:           ${portfolio_avg:,.0f}")


**Reading the result.** Strip `background_group` out of Jordan's waterfall entirely, and
the rest of his bars — mostly favorable, given decades of experience and zero at-fault
claims — would price him *below* the portfolio average, exactly what you'd expect for a
low-risk driver. `background_group` alone is the single largest bar in either direction,
and it's what drags him from a below-average price back up to an above-average one. His
good record isn't earning him the discount it should; one feature with no actuarial basis
is quietly cancelling it out. That's a specific, dollar-denominated answer to "why is
Jordan's price what it is" — not a black-box number.

One person is not proof of a systemic problem, though. It's possible Jordan is a fluke —
a strange corner of the data. Before flagging this to compliance, you need to check the
whole portfolio.

## 5. Beeswarm plot &mdash; is it just Jordan, or everyone?

A waterfall plot explains one person. A **beeswarm plot** explains *all* of them at once
— one dot per policyholder per feature, letting you see, at a glance, whether a pattern
is a one-off or something the model does systematically across the whole portfolio.

In [ ]:
#@title 📊 How to read a beeswarm plot (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
.bw{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border:1px solid #ecebff;border-radius:18px;padding:16px 18px;max-width:700px;color:#2c2350}
.bw-h{font-size:15.5px;font-weight:800;color:#3b2d6b;margin:0 0 10px}
.bw-row{display:flex;gap:11px;align-items:flex-start;padding:5px 0}
.bw-sw{flex:0 0 34px;height:16px;border-radius:5px;margin-top:2px}
.bw-tx{font-size:12px;color:#4a4360;line-height:1.4}
.bw-tx b{color:#2c2350}
</style>
<div class="bw">
  <div class="bw-h">Reading a SHAP beeswarm plot</div>
  <div class="bw-row"><div class="bw-sw" style="background:linear-gradient(90deg,#4d8fd6,#e0796d)"></div>
    <div class="bw-tx"><b>Each row is one feature</b>, ranked top-to-bottom by how much it typically
    moves predictions across the whole portfolio.</div></div>
  <div class="bw-row"><div class="bw-sw" style="background:linear-gradient(90deg,#4d8fd6,#e0796d)"></div>
    <div class="bw-tx"><b>Each dot is one policyholder.</b> Its horizontal position is that person's
    SHAP value for that feature: left of center pulls their price down, right pushes it up.</div></div>
  <div class="bw-row"><div class="bw-sw" style="background:linear-gradient(90deg,#4d8fd6,#e0796d)"></div>
    <div class="bw-tx"><b>Color is the feature's value</b> for that person. For a numeric feature
    (like <code>vehicle_value</code>) that's a real low-to-high scale: blue = low, red = high, and
    a row that goes cleanly blue-left / red-right is a clean, ordinary effect. For a
    <b>categorical</b> feature (like <code>background_group</code> or <code>urban_vs_rural</code>),
    color just marks *which category* — there's no real "low" or "high" — so instead of a smooth
    gradient you'll see a handful of distinct color clusters. That's expected, not a warning
    sign by itself; what's worth a second look is a category whose dots sit unusually far from
    center given how ordinary its members otherwise look.</div></div>
</div>
"""))


In [ ]:
#@title Portfolio-wide beeswarm plot { display-mode: "form" }
shap.plots.beeswarm(sv)


In [ ]:
#@title Portfolio-wide ranked importance { display-mode: "form" }
shap.plots.bar(sv)


In [ ]:
#@title 🎯 Quick check: one-off or systemic? (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
.q3{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:700px;color:#2c2350}
.q3-h{font-size:15.5px;font-weight:800;color:#3b2d6b;margin:0 0 8px}
.q3-q{font-size:13px;color:#4a4360;line-height:1.45;margin-bottom:12px}
.q3-opt{background:#fff;border:1.5px solid #e6e8ee;border-radius:12px;padding:10px 13px;
        margin-bottom:8px;font-size:12.5px;color:#3a3450;line-height:1.4;cursor:pointer;transition:border-color .15s}
.q3-opt:hover{border-color:#c9b8ec}
.q3-correct{border-color:#39b36a;background:#eef9f1}
.q3-wrong{border-color:#e0796d;background:#fdf1ef}
.q3-fb{margin-top:10px;padding:10px 13px;border-radius:12px;font-size:12.5px;line-height:1.45}
.q3-fb-good{background:#eef9f1;color:#1f6b3e;border:1px solid #b9e6c7}
.q3-fb-bad{background:#fdf1ef;color:#8a3d34;border:1px solid #f0c4bd}
</style>
<div class="q3">
  <div class="q3-h">🎯 Quick check</div>
  <div class="q3-q">Which statement is best supported by <code>background_group</code>'s row in the beeswarm above?</div>
  <div class="q3-opt" id="q3-opt-0" onclick="q3pick(0)">Real, but small &mdash; it ranks near the bottom of the 7 features.</div>
  <div class="q3-opt" id="q3-opt-1" onclick="q3pick(1)">Confined to a few unusual people like Jordan &mdash; most of its dots sit close to zero.</div>
  <div class="q3-opt" id="q3-opt-2" onclick="q3pick(2)">Systemic &mdash; it ranks 2nd of 7 features, and its dots split into two separated clusters rather than a smooth gradient, meaning most members of the affected group are pushed the same direction regardless of their other qualities.</div>
  <div class="q3-opt" id="q3-opt-3" onclick="q3pick(3)">Not visible in the beeswarm at all &mdash; you'd need the bar plot to see it.</div>
  <div class="q3-fb" id="q3-fb" style="display:none"></div>
</div>
<script>
(function(){
  var correct = 2;
  var feedback = [
    "Not quite — look again at its rank among the 7 rows; it's near the top, not the bottom.",
    "Not quite — scan the dots across the whole row, not just Jordan's: the split is wide, not a handful of outliers.",
    "Right — rank plus that two-cluster shape (instead of a smooth blue-to-red gradient) is the signature of a group-level, systemic effect, not a one-off.",
    "It is visible — that's exactly what the beeswarm's rank and shape show you; the bar plot only shows the ranking, not this pattern."
  ];
  window.q3pick = function(i){
    for(var j=0;j<4;j++){
      var el = document.getElementById('q3-opt-'+j);
      el.classList.remove('q3-correct','q3-wrong');
      if(j===correct) el.classList.add('q3-correct');
    }
    if(i!==correct) document.getElementById('q3-opt-'+i).classList.add('q3-wrong');
    var fb = document.getElementById('q3-fb');
    fb.style.display = 'block';
    fb.textContent = feedback[i];
    fb.className = 'q3-fb ' + (i===correct ? 'q3-fb-good' : 'q3-fb-bad');
  };
})();
</script>
"""))


### 🎯 Task — quantify background_group's effect, by group

The beeswarm and bar plot suggest `background_group` has a big, systemic impact — but only
visually so far. Pull out its actual SHAP values and average them within each group to put
a dollar figure on it.

In [ ]:
# 🎯 YOUR TURN — Exercise 3: quantify background_group's effect, by group.
#
# 💭 Think first: the beeswarm plot suggested background_group splits into two distinct
#    clusters rather than a smooth gradient. If that's really a group-level effect and not
#    just Jordan being unlucky, what should the *average* SHAP value look like for Group C
#    policyholders, versus Group A and Group B?
#
# 🎯 Implement: index the Explanation object with `sv[:, 'background_group']` to get
#    that one feature's values for every policyholder, take its `.values` attribute (the
#    raw numbers), and wrap it in `pd.Series(raw_numbers, index=X_test.index)`. Store it
#    in `bg_shap`.
bg_shap = ...
bg_group = df.loc[X_test.index, 'background_group']

print("Average SHAP contribution of background_group, by group:")
print(bg_shap.groupby(bg_group).mean().round(0).to_string())
print(f"\n{BIASED_GROUP} carries a positive push on essentially every policyholder who "
      f"belongs to it -- not just Jordan.")


**Reading the result.** `background_group` ranks 2nd of 7 features by average impact —
right behind the strongest legitimate predictor in the whole dataset, and ahead of every
other rating factor. Look closely at its row in the beeswarm: instead of the smooth
gradient you see for a continuous feature like `vehicle_value`, it splits into two clean,
separated clusters of dots. That split-cluster shape is often the visual signature of a
categorical proxy effect — one group of people getting pushed up almost as a block,
regardless of their color (their actual driving record) on every other row. This is not
one strange case. It is the model applying a group-level surcharge, learned from
historical data, with no basis in anyone's actual risk.

That turns into a concrete action: `background_group` needs to be pulled from the rating
model, and — because a model can reconstruct a dropped attribute from correlated
features — the remaining features should be re-checked for whether any of them are
quietly acting as a proxy for it, before this model is signed off.

### ⭐ A caution

SHAP explains what the **model** learned from the data it was given — not what is
actuarially fair, legally acceptable, or true about real-world risk. It is a detection
tool, not a certificate of fairness: it will just as faithfully explain a biased model as
an unbiased one. It also has a real blind spot around correlated features: when two
legitimate features overlap in what they tell the model (say, `years_licensed` and
`age` — older policyholders are, by construction, almost always the more experienced
ones), SHAP's fairness rule is to *split* the credit between them, even though only their
*combined* effect is really meaningful. A feature can end up looking unimportant on its
own simply because a correlated neighbor is quietly carrying half its story. Read
attributions as a strong lead worth investigating, not as a final verdict.

## 6. Apply it &mdash; hiring and compliance

You just used SHAP to catch a protected-attribute proxy driving an insurance price. The
same technique applies far beyond insurance.

> **Discussion.** Imagine your company is deploying an AI model to screen resumes for a
> job opening. How would you use SHAP to help ensure the model complies with labor law?

To get started, work through:

- Which features in a resume-screening model could be a protected attribute outright
  (age, gender), or a **proxy** for one (graduation year, a name, a zip code, a college)?
- If you ran a beeswarm plot across a large batch of past candidates, what pattern would
  make you suspicious?
- If you found that pattern — what would you actually do next, concretely?

## Summary

1. A SHAP value is the same thing as a fair bonus split: a feature's average marginal
   contribution to a prediction, across every order you could have revealed the features
   in. No formula needed to use or trust that idea.
2. A **waterfall plot** explains one decision, bar by bar, and always adds up exactly to
   the model's prediction — nothing hidden, nothing double-counted.
3. A **beeswarm plot** zooms out from one decision to a whole portfolio, turning "this one
   case looks off" into "this is (or isn't) a systemic pattern."
4. SHAP caught a protected-attribute proxy driving an individual insurance price and
   confirmed it was systemic across the portfolio — a real compliance workflow, not a toy
   example.
5. SHAP explains the **model**, not the truth or the law. It tells you where to look; it
   doesn't tell you the model — or the data it learned from — is fair.